# Лабораторная работа №1
## Разведочный анализ данных. Исследование и визуализация данных.

**Выполнил:** Зазовский В.Д.  
**Группа:** ИБМ3-64Б  
**Дата:** 2026

### Цель работы
Изучение различных методов визуализации данных.

### Описание набора данных
Используется набор данных **Iris** (ирисы Фишера) из scikit-learn — 150 образцов, 4 признака, 3 класса.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from math import pi

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
target_names = {i: name for i, name in enumerate(iris.target_names)}
df['species'] = df['target'].map(target_names)

print("Первые 10 строк:")
print(df.head(10))
print("\nРазмер:", df.shape)
print("\nКлассы:", df['species'].value_counts().to_dict())


In [ ]:
# Статистическое описание
print("=== СТАТИСТИЧЕСКОЕ ОПИСАНИЕ ===")
print(df[iris.feature_names].describe().round(3))
print("\n=== ПРОПУСКИ ===")
print(df.isnull().sum())


In [ ]:
# Гистограммы
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Распределение признаков по видам ирисов', fontsize=14, fontweight='bold')
features = iris.feature_names
colors = ['#e74c3c', '#2ecc71', '#3498db']
for idx, feature in enumerate(features):
    ax = axes[idx // 2][idx % 2]
    for s_idx, s_name in enumerate(iris.target_names):
        subset = df[df['species'] == s_name][feature]
        ax.hist(subset, alpha=0.6, label=s_name, color=colors[s_idx], bins=15, edgecolor='black')
    ax.set_xlabel(feature); ax.set_ylabel('Частота'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('/mnt/agents/output/tmo_labs/lab1/histograms.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# Boxplot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Boxplot: распределение признаков', fontsize=14, fontweight='bold')
for idx, feature in enumerate(features):
    ax = axes[idx // 2][idx % 2]
    sns.boxplot(data=df, x='species', y=feature, ax=ax, palette='Set2')
    ax.set_xlabel('Вид'); ax.set_ylabel(feature); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('/mnt/agents/output/tmo_labs/lab1/boxplots.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# Pairplot
pairplot = sns.pairplot(df[iris.feature_names + ['species']], hue='species', palette='Set2', diag_kind='kde', height=2.5)
pairplot.fig.suptitle('Парные диаграммы признаков Iris', y=1.02, fontsize=14, fontweight='bold')
plt.savefig('/mnt/agents/output/tmo_labs/lab1/pairplot.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# Корреляционная матрица
plt.figure(figsize=(10, 8))
corr = df[iris.feature_names].corr()
sns.heatmap(corr, annot=True, cmap='RdYlBu_r', center=0, fmt='.3f', square=True, linewidths=0.5)
plt.title('Корреляционная матрица', fontsize=14, fontweight='bold'); plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/lab1/correlation.png', dpi=150, bbox_inches='tight'); plt.show()
print(corr.round(3))


In [ ]:
# Scatter plots
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Диаграммы рассеяния', fontsize=14, fontweight='bold')
pairs = [('sepal length (cm)','sepal width (cm)'),('sepal length (cm)','petal length (cm)'),
         ('sepal length (cm)','petal width (cm)'),('sepal width (cm)','petal length (cm)'),
         ('sepal width (cm)','petal width (cm)'),('petal length (cm)','petal width (cm)')]
for idx, (x, y) in enumerate(pairs):
    ax = axes[idx//3][idx%3]
    for s_idx, s_name in enumerate(iris.target_names):
        subset = df[df['species']==s_name]
        ax.scatter(subset[x], subset[y], label=s_name, color=colors[s_idx], alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
    ax.set_xlabel(x); ax.set_ylabel(y); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('/mnt/agents/output/tmo_labs/lab1/scatter.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# Radar chart
categories = ['sepal\nlength','sepal\nwidth','petal\nlength','petal\nwidth']
N = len(categories)
means = df.groupby('species')[iris.feature_names].mean()
angles = [n/float(N)*2*pi for n in range(N)]
angles += angles[:1]
fig, ax = plt.subplots(figsize=(8,8), subplot_kw=dict(polar=True))
for i, (species, row) in enumerate(means.iterrows()):
    vals = row.values.tolist() + row.values.tolist()[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, label=species, color=colors[i])
    ax.fill(angles, vals, alpha=0.15, color=colors[i])
ax.set_xticks(angles[:-1]); ax.set_xticklabels(categories, size=11)
ax.set_title('Средние значения признаков', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3,1.1)); plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/lab1/radar.png', dpi=150, bbox_inches='tight'); plt.show()


### Выводы
1. Набор Iris содержит 150 образцов, 4 признака, 3 класса по 50 образцов. Пропусков нет.
2. Признаки petal length и petal width показывают наилучшую разделимость классов.
3. Сильная корреляция (r≈0.96) между petal length и petal width.
4. Класс setosa линейно отделим от других классов.
5. Набор данных хорошо подходит для построения моделей машинного обучения.
